# Cinema Phase 1 - Kaggle CPU

Use a fresh Kaggle Python session with Accelerator=None and Internet=On. Run cells in order and stop on errors. Run fixture creation and booking cells only once per session.

The backend is pinned to commit a51fb7f496f599243415d797f283baa2ecdff8bd. Backend, MySQL and the load generator share CPU/RAM. Generated credentials stay in memory: never print account dictionaries or APP_ENV. Saving a notebook does not preserve live processes or database state across sessions.

This imported copy clears old outputs, uses current-run variables for report downloads and enables Internet in metadata. Measurement code and business assertions are unchanged. Original JSON evidence is under docs/phase1/evidence; see benchmarks/kaggle/README.md for instructions. Apt package versions can change; each report records the Java/MySQL versions actually used.


In [ ]:
import os
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
REPO = WORK / "Cinema"
COMMIT = "a51fb7f496f599243415d797f283baa2ecdff8bd"

assert os.geteuid() == 0, "Cần quyền root để cài Java/MySQL"

memory = Path("/proc/meminfo").read_text()
print("CPU logic:", os.cpu_count())
print(next(x for x in memory.splitlines() if x.startswith("MemTotal:")))
print("Thư mục:", WORK)

def run_logged(command, filename, timeout=900, cwd=None):
    log = WORK / filename
    with log.open("w") as output:
        result = subprocess.run(
            command,
            cwd=cwd,
            env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"},
            stdout=output,
            stderr=subprocess.STDOUT,
            timeout=timeout
        )

    if result.returncode != 0:
        print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
        raise RuntimeError(f"Lệnh thất bại. Xem {log.name}")

    print(f"PASS: {filename}")

In [ ]:
run_logged(
    ["apt-get", "update", "-qq"],
    "setup-update.log"
)

run_logged(
    ["apt-get", "install", "-y",
     "openjdk-25-jdk-headless", "mysql-server", "git", "curl"],
    "setup-install.log"
)

JAVA_HOME = Path("/usr/lib/jvm/java-25-openjdk-amd64")
assert (JAVA_HOME / "bin/java").exists(), "Chưa tìm thấy Java 25"

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + ":" + os.environ["PATH"]

subprocess.run(["java", "-version"], check=True)
subprocess.run(["javac", "-version"], check=True)

run_logged(
    ["service", "mysql", "start"],
    "setup-mysql.log",
    timeout=120
)

subprocess.run(
    [
        "mysql", "--protocol=socket", "-u", "root",
        "-e", "SELECT VERSION() AS mysql_version, 1 AS connection_ok;"
    ],
    check=True,
    timeout=20
)

In [ ]:
if not REPO.exists():
    run_logged(
        ["git", "clone",
         "--branch", "feature/nghia/m12-delivery",
         "https://github.com/taesikwoo268/Cinema.git",
         str(REPO)],
        "setup-clone.log",
        timeout=240
    )

subprocess.run(
    ["git", "checkout", "--detach", COMMIT],
    cwd=REPO,
    check=True,
    timeout=30
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True
).strip()

assert actual_commit == COMMIT
print("PASS: Backend commit", actual_commit)

In [ ]:
import secrets

if "DB_PASSWORD" not in globals():
    DB_PASSWORD = secrets.token_hex(24)

sql = f"""
CREATE DATABASE IF NOT EXISTS cinema_kaggle
  CHARACTER SET utf8mb4 COLLATE utf8mb4_bin;

CREATE USER IF NOT EXISTS 'cinema_bench'@'localhost'
  IDENTIFIED BY '{DB_PASSWORD}';

ALTER USER 'cinema_bench'@'localhost'
  IDENTIFIED BY '{DB_PASSWORD}';

GRANT ALL PRIVILEGES ON cinema_kaggle.*
  TO 'cinema_bench'@'localhost';
"""

result = subprocess.run(
    ["mysql", "--protocol=socket", "-u", "root"],
    input=sql,
    capture_output=True,
    text=True,
    timeout=30
)

if result.returncode != 0:
    raise RuntimeError(
        result.stderr.replace(DB_PASSWORD, "[REDACTED]")
    )

APP_ENV = {
    **os.environ,
    "SPRING_PROFILES_ACTIVE": "mysql",
    "CINEMA_DB_URL": (
        "jdbc:mysql://127.0.0.1:3306/cinema_kaggle"
        "?connectionTimeZone=UTC&sslMode=DISABLED"
        "&allowPublicKeyRetrieval=true"
    ),
    "CINEMA_DB_USER": "cinema_bench",
    "CINEMA_DB_PASSWORD": DB_PASSWORD,
    "SERVER_ADDRESS": "127.0.0.1",
    "SERVER_PORT": "8080",
}

print("PASS: Database và tài khoản đã sẵn sàng.")

In [ ]:
run_logged(
    ["sh", "./mvnw", "-B", "-DskipTests", "package"],
    "backend-build.log",
    timeout=900,
    cwd=REPO
)

JAR = REPO / "target/cinema-0.1.0-SNAPSHOT.jar"
assert JAR.exists(), "Build chưa tạo file JAR"

print("PASS: Đã đóng gói backend.")

In [ ]:
import urllib.request
import urllib.error

BASE = "http://127.0.0.1:8080"
APP_LOG = WORK / "cinema-app.log"

if "cinema_process" in globals() and cinema_process.poll() is None:
    print("Backend đang chạy; giữ nguyên tiến trình.")
else:
    with APP_LOG.open("a") as output:
        cinema_process = subprocess.Popen(
            [str(JAVA_HOME / "bin/java"), "-jar", str(JAR)],
            cwd=REPO,
            env=APP_ENV,
            stdin=subprocess.DEVNULL,
            stdout=output,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

ready = False
deadline = time.monotonic() + 90

while time.monotonic() < deadline:
    if cinema_process.poll() is not None:
        break

    try:
        with urllib.request.urlopen(
            BASE + "/movies", timeout=2
        ) as response:
            if response.status == 200:
                print("PASS: GET /movies → HTTP 200")
                print("Response:", response.read().decode()[:200])
                ready = True
                break
    except (OSError, TimeoutError):
        pass

    time.sleep(1)

if not ready:
    tail = "\n".join(
        APP_LOG.read_text(errors="replace").splitlines()[-60:]
    )
    print(tail.replace(DB_PASSWORD, "[REDACTED]"))
    raise RuntimeError("Backend chưa sẵn sàng. Dừng tại ô này.")

result = subprocess.run(
    [
        "mysql", "--protocol=socket", "-u", "root",
        "--batch", "--skip-column-names", "cinema_kaggle",
        "-e",
        "SELECT version, success FROM flyway_schema_history "
        "WHERE version IS NOT NULL ORDER BY installed_rank;"
    ],
    capture_output=True,
    text=True,
    timeout=20,
    check=True
)

migrations = [
    line.split()
    for line in result.stdout.strip().splitlines()
]
assert migrations == [["1", "1"], ["2", "1"], ["3", "1"]], (
    f"Migrations không khớp: {migrations}"
)

print("PASS: Flyway V1, V2, V3 đều thành công.")
print("Backend PID:", cinema_process.pid)

In [ ]:
import uuid
from datetime import timedelta

def api(method, path, token=None, body=None, expected=200):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    request = urllib.request.Request(
        BASE + path,
        data=None if body is None else json.dumps(body).encode(),
        headers=headers,
        method=method
    )
    try:
        response = urllib.request.urlopen(request, timeout=20)
    except urllib.error.HTTPError as error:
        response = error

    with response:
        status = response.code
        content = response.read()

    assert status == expected, (
        f"{method} {path}: HTTP {status}, cần {expected}"
    )
    return json.loads(content) if content else None

def login(account):
    auth = api("POST", "/identity/login", body={
        "email": account["email"],
        "password": account["password"]
    })
    account["token"] = auth["accessToken"]
    return account["token"]

def new_account():
    account = {
        "email": f"kaggle-{uuid.uuid4()}@example.invalid",
        "password": secrets.token_hex(20)
    }
    api("POST", "/identity/register", body={
        **account, "displayName": "Kaggle test"
    }, expected=201)
    login(account)
    return account

def db_query(query):
    result = subprocess.run(
        ["mysql", "--protocol=socket", "-u", "root",
         "--batch", "--skip-column-names", "cinema_kaggle"],
        input=query, capture_output=True, text=True, timeout=20
    )
    if result.returncode != 0:
        raise RuntimeError("Truy vấn database thất bại.")
    return result.stdout.strip()

print("PASS: Các hàm hỗ trợ đã sẵn sàng.")

In [ ]:
assert "SMOKE" not in globals(), (
    "Đã tạo dữ liệu smoke trong phiên này; không chạy lại ô 8."
)

admin = new_account()
customer = new_account()
other_customer = new_account()

# Email được sinh ở trên; chỉ cấp quyền trong DB Kaggle.
db_query(
    "UPDATE user_records SET role='ADMIN' "
    f"WHERE email='{admin['email']}';"
)
login(admin)

def create(path, body):
    return api(
        "POST", path, admin["token"], body, expected=201
    )["id"]

cinema_id = create("/cinemas", {
    "name": "Kaggle smoke cinema", "address": "Kaggle"
})
hall_id = create("/halls", {
    "cinemaId": cinema_id, "name": "Smoke hall"
})
seat_id = create("/seats", {
    "hallId": hall_id, "row": "A", "number": 1, "type": "NORMAL"
})
movie_id = create("/movies", {
    "title": "Kaggle smoke movie",
    "description": "Functional check",
    "durationMinutes": 120, "ageRating": "PG"
})

start = datetime.now(timezone.utc) + timedelta(days=7)
screening_id = create("/screenings", {
    "movieId": movie_id,
    "hallId": hall_id,
    "startTime": start.isoformat(),
    "endTime": (start + timedelta(hours=2)).isoformat(),
    "basePrice": 12, "currency": "USD"
})

SMOKE = {
    "movie_id": movie_id,
    "screening_id": screening_id,
    "seat_id": seat_id
}
print("PASS: Đã tạo dữ liệu smoke riêng.")

In [ ]:
assert "SMOKE_RESULT" not in globals(), (
    "Smoke đã hoàn tất; không cần chạy lại."
)

token = login(customer)
foreign_token = login(other_customer)

seat_body = {
    "screeningId": SMOKE["screening_id"],
    "seatId": SMOKE["seat_id"]
}

# Không có token phải bị từ chối.
api("GET", "/tickets/me", expected=403)
api("POST", "/carts/seats", body=seat_body, expected=403)

# CUSTOMER không được tạo rạp.
api("POST", "/cinemas", token, {
    "name": "Denied", "address": "Denied"
}, expected=403)

# Luồng đặt vé.
api("POST", "/carts/seats", token, seat_body, expected=204)
checkout = api("POST", "/bookings/checkout", token)
assert len(checkout["bookingIds"]) == 1
booking_id = checkout["bookingIds"][0]

assert api(
    "POST", f"/payments/{booking_id}", token
)["successful"] is True

booking = api("GET", f"/bookings/{booking_id}", token)
assert booking["status"] == "CONFIRMED"

tickets = api("GET", "/tickets/me", token)
assert len(tickets) == 1
ticket = tickets[0]
assert ticket["bookingId"] == booking_id
assert ticket["screeningId"] == SMOKE["screening_id"]
assert ticket["seatId"] == SMOKE["seat_id"]
assert ticket["status"] == "VALID"

# Khách khác không được đọc vé hoặc trả tiền booking này.
api("GET", f"/tickets/{ticket['id']}", foreign_token, expected=403)
api("POST", f"/payments/{booking_id}", foreign_token, expected=403)

# Retry không phát thêm vé.
assert api(
    "POST", f"/payments/{booking_id}", token
)["successful"] is True
assert api("GET", "/tickets/me", token) == tickets

# Kiểm tra bản ghi thực tế sau retry.
sid = str(uuid.UUID(SMOKE["screening_id"]))
counts = [
    int(value) for value in db_query(f"""
SELECT
 (SELECT COUNT(*) FROM booking_records
  WHERE screening_id='{sid}' AND status='CONFIRMED'),
 (SELECT COUNT(*) FROM seat_reservation_records
  WHERE screening_id='{sid}'),
 (SELECT COUNT(*) FROM ticket_records
  WHERE screening_id='{sid}'),
 (SELECT COUNT(*) FROM payment_records p
  JOIN booking_records b ON p.booking_id=b.id
  WHERE b.screening_id='{sid}' AND p.status='SUCCESS');
""").split()
]
assert counts == [1, 1, 1, 1], f"Dữ liệu không khớp: {counts}"

SMOKE_RESULT = {
    "commit": COMMIT,
    "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "checks": [
        "GET/POST without token rejected",
        "Customer denied admin operation",
        "Checkout/payment/confirmed booking/valid ticket",
        "Cross-customer ticket/payment access rejected",
        "Payment retry preserves ticket",
        "Database counts after retry match"
    ],
    "counts": {
        "confirmed_bookings": counts[0],
        "reservations": counts[1],
        "tickets": counts[2],
        "successful_payments": counts[3]
    }
}

evidence = WORK / f"cinema-smoke-{time.time_ns()}.json"
evidence.write_text(
    json.dumps(SMOKE_RESULT, indent=2), encoding="utf-8"
)

print("PASS: Bảo mật, đặt vé, retry và database consistency.")
print("Đã lưu bằng chứng không chứa thông tin đăng nhập:", evidence)

In [ ]:
assert "LOAD" not in globals(), "Đã tạo dữ liệu tải; không chạy lại ô này."
assert cinema_process.poll() is None, "Backend đã dừng."

login(admin)

load_cinema = create("/cinemas", {
    "name": "Kaggle load cinema", "address": "Kaggle"
})
load_hall = create("/halls", {
    "cinemaId": load_cinema, "name": "Load hall"
})
load_movie = create("/movies", {
    "title": "Kaggle load movie",
    "description": "Load-test fixture",
    "durationMinutes": 120, "ageRating": "PG"
})

load_seats = [
    create("/seats", {
        "hallId": load_hall, "row": "A",
        "number": number, "type": "NORMAL"
    })
    for number in range(1, 31)
]

start = datetime.now(timezone.utc) + timedelta(days=8)
load_screening = create("/screenings", {
    "movieId": load_movie, "hallId": load_hall,
    "startTime": start.isoformat(),
    "endTime": (start + timedelta(hours=2)).isoformat(),
    "basePrice": 12, "currency": "USD"
})

LOAD = {
    "movie_id": load_movie,
    "hall_id": load_hall,
    "screening_id": load_screening,
    "seat_ids": load_seats
}

READ_PATHS = [
    ("movies", "/movies"),
    ("screenings", f"/screenings?movieId={load_movie}"),
    ("available-seats",
     f"/screenings/{load_screening}/available-seats")
]

# Các GET danh mục này công khai, không cần token.
assert any(x["id"] == load_movie for x in api("GET", "/movies"))
assert any(
    x["id"] == load_screening
    for x in api("GET", READ_PATHS[1][1])
)
assert len(api("GET", READ_PATHS[2][1])) == 30

print("PASS: Dữ liệu tải sẵn sàng; cả 3 endpoint trả đúng dữ liệu.")

In [ ]:
import requests
import threading
import math
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

def percentile(values, p):
    values = sorted(values)
    return round(values[max(0, math.ceil(len(values) * p / 100) - 1)], 2)

def read_load(users, seconds):
    gate = threading.Barrier(users + 1)
    timing = {}

    def worker():
        rows = []
        with requests.Session() as session:
            gate.wait()
            while time.perf_counter() < timing["end"]:
                for name, path in READ_PATHS:
                    if time.perf_counter() >= timing["end"]:
                        break

                    began = time.perf_counter()
                    status, ok, error = 0, False, ""

                    try:
                        with session.get(BASE + path, timeout=10) as response:
                            status = response.status_code
                            if status == 200:
                                ok = isinstance(response.json(), list)
                            if not ok:
                                error = f"HTTP_{status}_or_invalid_JSON"
                    except (requests.RequestException, ValueError) as exc:
                        error = type(exc).__name__

                    rows.append((
                        name,
                        (time.perf_counter() - began) * 1000,
                        status, ok, error
                    ))

                remaining = timing["end"] - time.perf_counter()
                if remaining > 0:
                    time.sleep(min(1, remaining))
        return rows

    with ThreadPoolExecutor(max_workers=users) as pool:
        futures = [pool.submit(worker) for _ in range(users)]
        began = time.perf_counter()
        timing["end"] = began + seconds
        gate.wait()
        rows = [row for future in futures for row in future.result()]
        elapsed = time.perf_counter() - began

    endpoints = {}
    for name, _ in READ_PATHS:
        selected = [row for row in rows if row[0] == name]
        assert selected, f"Không có mẫu cho {name}"
        durations = [row[1] for row in selected]

        endpoints[name] = {
            "requests": len(selected),
            "errors": sum(not row[3] for row in selected),
            "p50_ms": percentile(durations, 50),
            "p95_ms": percentile(durations, 95),
            "p99_ms": percentile(durations, 99),
            "statuses": dict(Counter(str(row[2]) for row in selected)),
            "error_types": dict(Counter(
                row[4] for row in selected if not row[3]
            ))
        }

    successful = sum(row[3] for row in rows)
    return {
        "users": users,
        "duration_seconds": round(elapsed, 2),
        "requests": len(rows),
        "successful_requests_per_second": round(successful / elapsed, 2),
        "error_percent": round(100 * (len(rows) - successful) / len(rows), 3),
        "endpoints": endpoints
    }

print("PASS: Bộ đo sẵn sàng.")

In [ ]:
REPORT_PATH = WORK / f"cinema-read-load-{time.time_ns()}.json"

READ_REPORT = {
    "commit": COMMIT,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "logical_cpus": os.cpu_count(),
    "memory_total": next(
        line for line in Path("/proc/meminfo").read_text().splitlines()
        if line.startswith("MemTotal:")
    ),
    "java": subprocess.run(
        [str(JAVA_HOME / "bin/java"), "-version"],
        capture_output=True, text=True, check=True
    ).stderr.strip(),
    "mysql": db_query("SELECT VERSION();"),
    "think_time_seconds": 1,
    "request_timeout_seconds": 10,
    "paths": READ_PATHS,
    "fixture": LOAD,
    "notes": [
        "Backend, MySQL and load generator share the Kaggle CPU runtime.",
        "Small dataset; smoke-test catalog remains in database.",
        "Public read endpoints; connection pooling; no automatic retries.",
        "Latency includes failed requests. Status 0 means no HTTP response.",
        "Exploratory stop thresholds: errors >= 1% or endpoint p95 >= 2000 ms."
    ],
    "status": "running",
    "samples": []
}

def save_read_report():
    REPORT_PATH.write_text(
        json.dumps(READ_REPORT, indent=2), encoding="utf-8"
    )

def check_ready():
    assert cinema_process.poll() is None, "Tiến trình backend đã dừng."
    for name, path in READ_PATHS:
        assert isinstance(api("GET", path), list), f"{name}: JSON không hợp lệ"

save_read_report()

try:
    check_ready()
    print("Warm-up 15 giây...", flush=True)
    warmup = read_load(5, 15)
    READ_REPORT["warmup"] = warmup
    assert warmup["error_percent"] == 0, "Warm-up lỗi; dừng để kiểm tra."

    for users in [10, 25, 50, 100]:
        check_ready()
        print(f"Đang đo {users} người dùng — 60 giây...", flush=True)

        sample = read_load(users, 60)
        sample["backend_exit_code_after"] = cinema_process.poll()
        READ_REPORT["samples"].append(sample)
        save_read_report()
        print(json.dumps(sample, indent=2), flush=True)

        check_ready()
        worst_p95 = max(
            endpoint["p95_ms"]
            for endpoint in sample["endpoints"].values()
        )
        if sample["error_percent"] >= 1 or worst_p95 >= 2000:
            READ_REPORT["status"] = "stopped_at_threshold"
            print("Dừng tăng tải tại ngưỡng khảo sát.")
            break
    else:
        READ_REPORT["status"] = "completed"

except Exception as exc:
    READ_REPORT["status"] = "interrupted_or_failed"
    READ_REPORT["failure_type"] = type(exc).__name__
    raise

finally:
    save_read_report()
    print("Báo cáo:", REPORT_PATH)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64
import html

files = [
    REPORT_PATH,
    evidence
]

for file in files:
    if not file.exists():
        print("Không tìm thấy file:", file.name)
        continue

    encoded = base64.b64encode(file.read_bytes()).decode()
    name = html.escape(file.name, quote=True)
    display(HTML(
        f'<p><a download="{name}" '
        f'href="data:application/json;base64,{encoded}">'
        f'Tải {name}</a></p>'
    ))

In [ ]:
assert cinema_process.poll() is None, "Backend đã dừng."
assert "WRITE_FIXTURE" not in globals(), "Đã tạo dữ liệu; không chạy lại."

login(admin)

write_hall = create("/halls", {
    "cinemaId": load_cinema,
    "name": f"Concurrency {uuid.uuid4().hex[:8]}"
})
write_seats = [
    create("/seats", {
        "hallId": write_hall,
        "row": "A", "number": n, "type": "NORMAL"
    })
    for n in range(1, 31)
]

start = datetime.now(timezone.utc) + timedelta(days=10)

def create_test_screening(offset_hours):
    begins = start + timedelta(hours=offset_hours)
    return create("/screenings", {
        "movieId": LOAD["movie_id"],
        "hallId": write_hall,
        "startTime": begins.isoformat(),
        "endTime": (begins + timedelta(hours=2)).isoformat(),
        "basePrice": 12, "currency": "USD"
    })

booking_screening = create_test_screening(0)
contention_screening = create_test_screening(3)

print("Đang tạo 30 khách đặt ghế riêng...", flush=True)
booking_customers = [new_account() for _ in range(30)]

print("Đang tạo 10 khách tranh ghế...", flush=True)
contention_customers = [new_account() for _ in range(10)]

WRITE_FIXTURE = {
    "booking_screening": booking_screening,
    "contention_screening": contention_screening,
    "seat_ids": write_seats
}

print("PASS: Đã chuẩn bị dữ liệu; không xuất thông tin đăng nhập.")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading

assert "WRITE_REPORT" not in globals(), "Đã chạy; không chạy lại cùng dữ liệu."
assert cinema_process.poll() is None, "Backend đã dừng."

# Làm mới token nếu có khoảng chờ sau ô 13.
for account in booking_customers + contention_customers:
    login(account)

def try_add_seat(account, screening, seat):
    request = urllib.request.Request(
        BASE + "/carts/seats",
        data=json.dumps({
            "screeningId": screening, "seatId": seat
        }).encode(),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {account['token']}"
        },
        method="POST"
    )
    try:
        response = urllib.request.urlopen(request, timeout=20)
    except urllib.error.HTTPError as error:
        response = error
    with response:
        response.read()
        return response.code

def finish_booking(account, screening, seat):
    token = account["token"]
    checkout = api("POST", "/bookings/checkout", token)
    assert len(checkout["bookingIds"]) == 1
    booking_id = checkout["bookingIds"][0]

    paid = api("POST", f"/payments/{booking_id}", token)
    assert paid["successful"] is True
    assert api(
        "GET", f"/bookings/{booking_id}", token
    )["status"] == "CONFIRMED"

    tickets = api("GET", "/tickets/me", token)
    assert len(tickets) == 1
    ticket = tickets[0]
    assert ticket["bookingId"] == booking_id
    assert ticket["screeningId"] == screening
    assert ticket["seatId"] == seat
    assert ticket["status"] == "VALID"

    return {"status": "success", "booking_id": booking_id}

def attempt(account, screening, seat, gate=None):
    try:
        if gate is not None:
            gate.wait(timeout=30)

        status = try_add_seat(account, screening, seat)
        if status == 409:
            return {"status": "conflict", "http": 409}
        assert status == 204, f"Add seat: HTTP {status}"

        return finish_booking(account, screening, seat)
    except Exception as error:
        return {
            "status": "error",
            "type": type(error).__name__,
            "message": str(error)[:300]
        }

def database_counts(screening):
    sid = str(uuid.UUID(screening))
    values = db_query(f"""
SELECT
 (SELECT COUNT(*) FROM booking_records WHERE screening_id='{sid}'),
 (SELECT COUNT(*) FROM booking_records
  WHERE screening_id='{sid}' AND status='CONFIRMED'),
 (SELECT COUNT(*) FROM seat_reservation_records WHERE screening_id='{sid}'),
 (SELECT COUNT(*) FROM ticket_records WHERE screening_id='{sid}'),
 (SELECT COUNT(*) FROM payment_records p
  JOIN booking_records b ON p.booking_id=b.id
  WHERE b.screening_id='{sid}'),
 (SELECT COUNT(*) FROM payment_records p
  JOIN booking_records b ON p.booking_id=b.id
  WHERE b.screening_id='{sid}' AND p.status='SUCCESS');
""")
    names = [
        "bookings", "confirmed_bookings", "reservations",
        "tickets", "payments", "successful_payments"
    ]
    return dict(zip(names, map(int, values.split())))

WRITE_PATH = WORK / f"cinema-write-load-{time.time_ns()}.json"
WRITE_REPORT = {
    "commit": COMMIT,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "running",
    "note": "Finite concurrency checks, not sustained booking capacity.",
}

def save_write_report():
    WRITE_PATH.write_text(
        json.dumps(WRITE_REPORT, indent=2), encoding="utf-8"
    )

save_write_report()

try:
    print("30 khách / 30 ghế, tối đa 5 khách đồng thời...", flush=True)
    with ThreadPoolExecutor(max_workers=5) as pool:
        futures = [
            pool.submit(
                attempt, account,
                WRITE_FIXTURE["booking_screening"],
                WRITE_FIXTURE["seat_ids"][i]
            )
            for i, account in enumerate(booking_customers)
        ]
        separate = [future.result() for future in futures]

    WRITE_REPORT["separate_seats"] = {
        "outcomes": separate,
        "database": database_counts(WRITE_FIXTURE["booking_screening"])
    }
    save_write_report()

    print("10 khách tranh một ghế...", flush=True)
    gate = threading.Barrier(10)
    with ThreadPoolExecutor(max_workers=10) as pool:
        futures = [
            pool.submit(
                attempt, account,
                WRITE_FIXTURE["contention_screening"],
                WRITE_FIXTURE["seat_ids"][0], gate
            )
            for account in contention_customers
        ]
        contention = [future.result() for future in futures]

    WRITE_REPORT["same_seat"] = {
        "outcomes": contention,
        "database": database_counts(WRITE_FIXTURE["contention_screening"])
    }
    save_write_report()

    assert all(r["status"] == "success" for r in separate)
    assert len({r["booking_id"] for r in separate}) == 30
    assert all(
        n == 30 for n in WRITE_REPORT["separate_seats"]["database"].values()
    )

    assert sum(r["status"] == "success" for r in contention) == 1
    assert sum(r["status"] == "conflict" for r in contention) == 9
    assert all(
        n == 1 for n in WRITE_REPORT["same_seat"]["database"].values()
    )

    WRITE_REPORT["status"] = "passed"
    print("PASS: 30 khách đặt 30 ghế, có đúng 30 booking/payment/ticket.")
    print("PASS: Tranh một ghế — 1 thành công, 9 HTTP 409.")
    print("PASS: Database không có booking/payment/ticket thừa.")

except Exception:
    WRITE_REPORT["status"] = "failed"
    raise

finally:
    save_write_report()
    print("Báo cáo:", WRITE_PATH)

In [ ]:
from IPython.display import HTML, display
import base64

file = WRITE_PATH
encoded = base64.b64encode(file.read_bytes()).decode()

display(HTML(
    f'<a download="{file.name}" '
    f'href="data:application/json;base64,{encoded}">'
    f'Tải {file.name}</a>'
))